In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

# Read bronze
df = spark.table("bronze_student_info")

silver_students = df \
    .dropDuplicates(["id_student"]) \
    .withColumnRenamed("id_student", "student_id") \
    .withColumnRenamed("code_module", "course_id") \
    .withColumnRenamed("code_presentation", "semester") \
    .withColumn("age_band", F.col("age_band").cast("string")) \
    .withColumn("gender", F.col("gender").cast("string")) \
    .withColumn("imd_band", F.col("imd_band").cast("string")) \
    .withColumn("highest_education", F.col("highest_education").cast("string")) \
    .withColumn("num_of_prev_attempts", F.col("num_of_prev_attempts").cast(IntegerType())) \
    .withColumn("studied_credits", F.col("studied_credits").cast(IntegerType())) \
    .withColumn("disability", F.col("disability").cast("string")) \
    .withColumn("final_result", F.col("final_result").cast("string")) \
    .withColumn("is_at_risk", F.when(
        F.col("final_result").isin("Withdrawn", "Fail"), 1
    ).otherwise(0)) \
    .withColumn("_processed_at", F.current_timestamp()) \
    .select(
        "student_id", "course_id", "semester", "gender", "age_band",
        "imd_band", "highest_education", "num_of_prev_attempts",
        "studied_credits", "disability", "final_result", "is_at_risk",
        "_processed_at"
    )

silver_students.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_students")

print(f"silver_students: {silver_students.count():,} rows")

StatementMeta(, 8b339c1b-edaa-4649-9e31-f77a0cc8f6db, 3, Finished, Available, Finished, False)

silver_students: 28,785 rows


In [2]:
df = spark.table("bronze_student_assessment")
assessments_meta = spark.table("bronze_assessments")

silver_assessments = df \
    .withColumnRenamed("id_student", "student_id") \
    .withColumnRenamed("id_assessment", "assessment_id") \
    .withColumn("score", F.col("score").cast(FloatType())) \
    .withColumn("date_submitted", F.col("date_submitted").cast(IntegerType())) \
    .withColumn("is_banked", F.col("is_banked").cast(IntegerType())) \
    .withColumn("submission_status", F.when(
        F.col("score").isNull(), "not_submitted"
    ).otherwise("submitted")) \
    .withColumn("score_filled", F.coalesce(F.col("score"), F.lit(0.0))) \
    .dropDuplicates(["student_id", "assessment_id"]) \
    .withColumn("_processed_at", F.current_timestamp())

# Join assessment metadata to get due date and weight
assessments_meta_clean = assessments_meta \
    .withColumnRenamed("id_assessment", "assessment_id") \
    .withColumnRenamed("code_module", "course_id") \
    .withColumnRenamed("code_presentation", "semester") \
    .withColumn("date", F.col("date").cast(IntegerType())) \
    .withColumn("weight", F.col("weight").cast(FloatType())) \
    .select("assessment_id", "course_id", "semester", "assessment_type", "date", "weight")

silver_assessments = silver_assessments.join(
    assessments_meta_clean, on="assessment_id", how="left"
)

# Submission delay = days submitted after due date (positive = late)
silver_assessments = silver_assessments.withColumn(
    "submission_delay_days",
    F.col("date_submitted") - F.col("date")
)

silver_assessments.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_assessments")

print(f"silver_assessments: {silver_assessments.count():,} rows")

StatementMeta(, 8b339c1b-edaa-4649-9e31-f77a0cc8f6db, 4, Finished, Available, Finished, False)

silver_assessments: 173,912 rows


In [3]:
from pyspark.sql.window import Window

df = spark.table("bronze_student_vle")
vle_meta = spark.table("bronze_vle")

# Rename in vle_meta BEFORE join
vle_meta_clean = vle_meta \
    .withColumnRenamed("code_module", "course_id") \
    .select("id_site", "course_id", "activity_type")

# Rename code_module in df BEFORE join so both sides use course_id
silver_vle = df \
    .withColumnRenamed("id_student", "student_id") \
    .withColumnRenamed("code_module", "course_id") \
    .withColumnRenamed("code_presentation", "semester") \
    .withColumn("date", F.col("date").cast(IntegerType())) \
    .withColumn("sum_click", F.col("sum_click").cast(IntegerType())) \
    .join(vle_meta_clean, on=["id_site", "course_id"], how="left") \
    .withColumn("_processed_at", F.current_timestamp()) \
    .select(
        "student_id", "course_id", "semester", "id_site",
        "activity_type", "date", "sum_click", "_processed_at"
    )

# Session = group consecutive days of activity per student per course
# A new session starts if gap > 3 days
window_spec = Window.partitionBy("student_id", "course_id").orderBy("date")
silver_vle = silver_vle \
    .withColumn("prev_date", F.lag("date", 1).over(window_spec)) \
    .withColumn("day_gap", F.col("date") - F.col("prev_date")) \
    .withColumn("new_session_flag", F.when(
        (F.col("day_gap") > 3) | F.col("day_gap").isNull(), 1
    ).otherwise(0)) \
    .withColumn("session_id", F.sum("new_session_flag").over(
        window_spec.rowsBetween(Window.unboundedPreceding, 0)
    ))

silver_vle.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_vle_activity")

print(f"silver_vle_activity: {silver_vle.count():,} rows")

StatementMeta(, 8b339c1b-edaa-4649-9e31-f77a0cc8f6db, 5, Finished, Available, Finished, False)

silver_vle_activity: 10,655,280 rows


In [4]:
# OULAD has no instructor data.
# We derive one instructor per course module explicitly.
# This is synthetic and labeled as such.

instructor_data = [
    ("AAA", "INST_001", "Dr. A. Rahman",   "Mathematics"),
    ("BBB", "INST_002", "Dr. B. Clarke",   "Computing"),
    ("CCC", "INST_003", "Dr. C. Patel",    "Science"),
    ("DDD", "INST_004", "Dr. D. Okonkwo",  "Social Studies"),
    ("EEE", "INST_005", "Dr. E. Nguyen",   "Engineering"),
    ("FFF", "INST_006", "Dr. F. Martinez", "Humanities"),
    ("GGG", "INST_007", "Dr. G. Smith",    "Business"),
]

instructor_schema = ["course_id", "instructor_id", "instructor_name", "department"]

silver_instructors = spark.createDataFrame(instructor_data, schema=instructor_schema) \
    .withColumn("_is_synthetic", F.lit(True)) \
    .withColumn("_processed_at", F.current_timestamp())

silver_instructors.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_instructors")

print(f"silver_instructors: {silver_instructors.count():,} rows")
silver_instructors.show(truncate=False)

StatementMeta(, 8b339c1b-edaa-4649-9e31-f77a0cc8f6db, 6, Finished, Available, Finished, False)

silver_instructors: 7 rows
+---------+-------------+---------------+--------------+-------------+--------------------------+
|course_id|instructor_id|instructor_name|department    |_is_synthetic|_processed_at             |
+---------+-------------+---------------+--------------+-------------+--------------------------+
|AAA      |INST_001     |Dr. A. Rahman  |Mathematics   |true         |2026-04-16 20:29:37.117135|
|BBB      |INST_002     |Dr. B. Clarke  |Computing     |true         |2026-04-16 20:29:37.117135|
|CCC      |INST_003     |Dr. C. Patel   |Science       |true         |2026-04-16 20:29:37.117135|
|DDD      |INST_004     |Dr. D. Okonkwo |Social Studies|true         |2026-04-16 20:29:37.117135|
|EEE      |INST_005     |Dr. E. Nguyen  |Engineering   |true         |2026-04-16 20:29:37.117135|
|FFF      |INST_006     |Dr. F. Martinez|Humanities    |true         |2026-04-16 20:29:37.117135|
|GGG      |INST_007     |Dr. G. Smith   |Business      |true         |2026-04-16 20:29:37.1

In [7]:
df = spark.table("bronze_lms_events_raw")

silver_events = df \
    .withColumn("student_id", F.col("student_id").cast(IntegerType())) \
    .withColumn("score", F.col("score").cast(FloatType())) \
    .withColumn("duration_sec", F.col("duration_sec").cast(IntegerType())) \
    .withColumn("event_ts", F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_date", F.to_date(F.col("event_ts"))) \
    .withColumn("event_hour", F.hour(F.col("event_ts"))) \
    .filter(F.col("student_id").isNotNull()) \
    .filter(F.col("event_type").isNotNull()) \
    .filter(F.col("event_ts").isNotNull()) \
    .dropDuplicates(["session_id", "event_type", "student_id"]) \
    .withColumn("_processed_at", F.current_timestamp()) \
    .select(
        "student_id", "course_id", "instructor_id", "event_type",
        "event_ts", "event_date", "event_hour", "session_id",
        "duration_sec", "score", "status", "source", "_processed_at"
    )

silver_events.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_lms_events")

print(f"silver_lms_events: {silver_events.count():,} rows")

StatementMeta(, 8b339c1b-edaa-4649-9e31-f77a0cc8f6db, 9, Finished, Available, Finished, False)

silver_lms_events: 415 rows


In [10]:
silver_tables = {
    "silver_students":      ("student_id",    28000),  
    "silver_assessments":   ("student_id",   150000),
    "silver_vle_activity":  ("student_id",  5000000),
    "silver_instructors":   ("course_id",         7),
    "silver_lms_events":    ("student_id",        1),
}

print("--- Silver Validation ---")
for table, (key_col, min_rows) in silver_tables.items():
    df = spark.table(table)
    count = df.count()
    nulls = df.filter(F.col(key_col).isNull()).count()
    row_status = "✅" if count >= min_rows else "❌"
    null_status = "✅" if nulls == 0 else f"⚠️  {nulls} nulls"
    print(f"{row_status} {table}: {count:,} rows | key nulls: {null_status}")

StatementMeta(, 8b339c1b-edaa-4649-9e31-f77a0cc8f6db, 12, Finished, Available, Finished, False)

--- Silver Validation ---
✅ silver_students: 28,785 rows | key nulls: ✅
✅ silver_assessments: 173,912 rows | key nulls: ✅
✅ silver_vle_activity: 10,655,280 rows | key nulls: ✅
✅ silver_instructors: 7 rows | key nulls: ✅
✅ silver_lms_events: 415 rows | key nulls: ✅
